## Importações

In [ ]:
!pip3 install gymnasium

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 5.4 MB/s eta 0:00:00


In [ ]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
env = gym.make("CartPole-v1")

# configurar matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

In [ ]:
# Se a GPU for usada
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

## Replay Memory

In [ ]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))

In [ ]:
class ReplayMemory(object):
    """Classe para armazenar e amostrar transições de memória de repetição"""

    def __init__(self, capacity):
        # Inicializa uma deque (fila de duas extremidades) com um tamanho máximo de `capacity`
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Salva uma transição"""
        # Adiciona uma nova transição à memória
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        """Amostra um lote de transições aleatoriamente"""
        # Retorna um número especificado (batch_size) de transições aleatórias da memória
        return random.sample(self.memory, batch_size)

    def __len__(self):
        """Retorna o tamanho atual da memória"""
        # Retorna o número de transições atualmente armazenadas na memória
        return len(self.memory)

## Model

In [ ]:
class DQN(nn.Module):
    """Rede Neural Profunda para a DQN (Deep Q-Network)"""

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        # Primeira camada totalmente conectada (input para hidden)
        self.layer1 = nn.Linear(n_observations, 128)
        # Segunda camada totalmente conectada (hidden para hidden)
        self.layer2 = nn.Linear(128, 128)
        # Terceira camada totalmente conectada (hidden para output)
        self.layer3 = nn.Linear(128, n_actions)

    # Chamado com um elemento para determinar a próxima ação, ou um lote durante a otimização
    # Retorna tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        # Aplicar a função de ativação ReLU após a primeira camada
        x = F.relu(self.layer1(x))
        # Aplicar a função de ativação ReLU após a segunda camada
        x = F.relu(self.layer2(x))
        # Passar pela terceira camada sem ativação para obter os valores Q para cada ação
        return self.layer3(x)

## Training

In [ ]:
BATCH_SIZE = 128 # BATCH_SIZE é o número de transições amostradas do buffer de reprodução
GAMMA = 0.99     # GAMMA é o fator de desconto
EPS_START = 0.9  # EPS_START é o valor inicial do épsilon
EPS_END = 0.05   # EPS_END é o valor final do épsilon
EPS_DECAY = 1000 # EPS_DECAY controla a taxa de decaimento exponencial do épsilon, maior significa um decaimento mais lento
TAU = 0.005      # TAU é a taxa de atualização da rede alvo
LR = 1e-4        # LR é a taxa de aprendizado do otimizador ``AdamW``

In [ ]:
n_actions = env.action_space.n # Obtenha o número de ações do espaço de ação da academia
state, info = env.reset() # Obtenha o número de observações de estado
n_observations = len(state)

In [ ]:
# Cria a rede neural de política e move para o dispositivo especificado (CPU ou GPU)
policy_net = DQN(n_observations, n_actions).to(device)

# Cria a rede neural alvo e move para o dispositivo especificado
target_net = DQN(n_observations, n_actions).to(device)

# Inicializa a rede neural alvo com os mesmos pesos da rede de política
target_net.load_state_dict(policy_net.state_dict())

<All keys matched successfully>

In [ ]:
BUFFER_SIZE = int(1e5)  # Tamanho do buffer de replay
memory = ReplayMemory(BUFFER_SIZE) # Inicializa a memória de replay com a capacidade especificada

# Inicializa o otimizador AdamW com os parâmetros da policy_net, taxa de aprendizado LR, e amsgrad habilitado
# amsgrad=True: Habilita a variante AMSGrad do Adam, que pode proporcionar maior estabilidade em alguns casos
optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)

In [ ]:
steps_done = 0  # Inicializa o contador de passos

def select_action(state):
    global steps_done  # Usa a variável global steps_done para manter o controle do número de passos realizados
    sample = random.random()  # Gera um número aleatório entre 0 e 1
    # Calcula o valor de epsilon (probabilidade de escolher uma ação aleatória)
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1  # Incrementa o contador de passos

    # Verifica se o número aleatório é maior que o epsilon
    if sample > eps_threshold:
        # Se o número aleatório for maior que o epsilon, selecione a ação com maior recompensa esperada (ação "greedy")
        with torch.no_grad():
            # policy_net(state) retorna um tensor de valores Q para cada ação
            # t.max(1) retorna o valor máximo em cada linha, max(1) retorna (valores, indices)
            # .indices.view(1, 1) retorna o índice da ação com maior valor Q
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        # Caso contrário, selecione uma ação aleatória
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

In [ ]:
# Lista para armazenar as durações dos episódios
episode_durations = []

def plot_durations(show_result=False):
    plt.figure(1)  # Cria ou seleciona a figura número 1
    durations_t = torch.tensor(episode_durations, dtype=torch.float)  # Converte as durações dos episódios em um tensor PyTorch
    if show_result:
        plt.title('Result')  # Define o título do gráfico como 'Result' se show_result for True
    else:
        plt.clf()  # Limpa a figura atual
        plt.title('Training...')  # Define o título do gráfico como 'Training...' se show_result for False
    plt.xlabel('Episode')  # Define o rótulo do eixo x como 'Episode'
    plt.ylabel('Duration')  # Define o rótulo do eixo y como 'Duration'
    plt.plot(durations_t.numpy())  # Plota as durações dos episódios

    # Calcula e plota a média das durações dos últimos 100 episódios
    if len(durations_t) >= 100:
        # Cria uma janela deslizante de tamanho 100 e calcula a média
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        # Concatena zeros no início para alinhar com os episódios iniciais
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())  # Plota as médias

    plt.pause(0.001)  # Pausa um pouco para que os gráficos sejam atualizados

    # Se estiver rodando em um notebook IPython, exibe e atualiza os gráficos
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())  # Exibe o gráfico atual
            display.clear_output(wait=True)  # Limpa a saída para atualizar o gráfico
        else:
            display.display(plt.gcf())  # Exibe o gráfico final

## Training Loop

In [ ]:
def optimize_model():
    # Verifica se há transições suficientes no replay memory para formar um batch
    if len(memory) < BATCH_SIZE:
        return  # Se não houver, retorna sem fazer nada

    # Amostra um batch de transições do replay memory
    transitions = memory.sample(BATCH_SIZE)

    # Transpõe o batch para obter um Transition de batch-arrays em vez de um batch-array de Transitions
    # Lembre-se que Transition tem forma: Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))
    # Isso facilita o trabalho com os dados em PyTorch
    batch = Transition(*zip(*transitions))

    # Cria uma máscara de estados não finais e concatena os elementos do batch
    # Um estado final é aquele em que a simulação terminou
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                            batch.next_state)), device=device, dtype=torch.bool)

    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])

    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Computa Q(s_t, a) - o modelo computa Q(s_t), então selecionamos as colunas das ações tomadas
    # Estas são as ações que teriam sido tomadas para cada estado do batch de acordo com policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Computa os retornos para todos os próximos estados.
    # Valores esperados das ações para non_final_next_states são computados com  base no target_net "mais antigo"
    # Selecionando a melhor recompensa com max(1).values
    # Isso é mesclado com base na máscara, de modo que teremos o valor do estado esperado ou 0 caso o estado fosse final
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values

    # Computa os valores Q esperados
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Computa a perda Huber
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    ################# Otimiza o modelo #################
    optimizer.zero_grad()  # Zera os gradientes acumulados
    loss.backward()        # Computa os gradientes
    # Clipping dos gradientes para evitar explosão de gradientes
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()       # Atualiza os pesos do modelo

In [ ]:
%%time

if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = 600  # Define número de episódios para hardware acelerado
else:
    num_episodes = 50  # Define número de episódios para hardware não acelerado


for i_episode in range(num_episodes):
    # Inicializa o ambiente e obtém o estado inicial
    state, _ = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    for t in count():
        # Seleciona uma ação com base no estado atual usando a política epsilon-greedy
        action = select_action(state)
        observation, reward, terminated, truncated, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        done = terminated or truncated

        if terminated:
            next_state = None  # Define próximo estado como None se o episódio terminou
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        memory.push(state, action, next_state, reward) # Armazena a transição na memória

        state = next_state # Move para o próximo estado

        optimize_model() # Executa uma etapa de otimização

        # Atualização suave dos pesos da rede alvo
        # θ′ ← τ θ + (1 − τ)θ′
        # Atualiza suavemente os pesos da rede alvo com base nos pesos da rede de política
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()

        for key in policy_net_state_dict:
            # Combina os pesos da rede de política (policy_net) e da rede alvo (target_net)
            # Usando a fórmula: θ′ ← τ θ + (1 − τ)θ′
            # Onde τ é o fator de suavização (TAU)
            # Esta operação ajuda a evitar grandes oscilações nos valores Q
            target_net_state_dict[key] = policy_net_state_dict[key] * TAU + target_net_state_dict[key] * (1 - TAU)

        target_net.load_state_dict(target_net_state_dict) # Carrega os novos pesos suavizados de volta na rede alvo

        if done:
            # Armazena a duração do episódio e plota os resultados
            episode_durations.append(t + 1)
            plot_durations()
            break

print('Complete')
plot_durations(show_result=True)
plt.ioff()
plt.show()